<a href="https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sizan99/ml-pipeline/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

* **Unit of analysis:** One row = one content item on a specific day for a specific client (Page-Day grain).
* **Table used:** `fact_content_daily_performance` (Hugging Face Warehouse).
* **Time window:** I'm analyzing a single mid-panel month (**March 2026**, `month=2026-03`). I'm excluding the final month (June 2026) because that acts as the sealed test set.

In [23]:
# Verified in Section 3 queries below.

## 2. Fields: feature / label / context / excluded

* **Features:** `cpc`, `search_volume`, `word_count`, `avg_position`, `impressions_prev30`. These are all safe to use because they are knowable *before* the prediction moment.
* **Label / proxy:** `clicks` (the future outcome I want to maximize or predict).
* **Context:** `content_id`, `client_id`, `report_date`, `main_intent`. (Used for grouping and reading, never for the model).
* **Excluded:** `trend_pct`. *Why excluded?* Because it is calculated using future data. If I use it, the model will just memorize the rule (leakage) rather than learning from the features.

In [24]:
# Verified in Section 3 queries below.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [25]:
import pandas as pd
from google.colab import userdata

# Load token from Colab Secrets
hf_token = userdata.get('HF_TOKEN')
print("Loading March 2026 dataset from Hugging Face...")
dataset_url = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"
df = pd.read_parquet(dataset_url, storage_options={"token": hf_token})
print("Loaded successfully!\n")

# 1. Verify Grain (Page-Day)
grain_max = df.groupby(['client_hash_id', 'content_hash_id', 'report_date']).size().max()
print("Query 1: Grain verification (Page-Day)")
print(f"Result: Max rows per client-content-date is {grain_max} (if 1, grain holds perfectly!)\n")

# 2. Row count and Date Span
print("Query 2: Row count and Date Span")
print(f"Result: {len(df):,} rows, from {df['report_date'].min()} to {df['report_date'].max()}\n")

# 3. Availability check
survived_rows = len(df[df['ga4_data_available'] == True])
print("Query 3: Availability (IS TRUE)")
print(f"Result: {survived_rows:,} rows survive the GA4 availability filter (out of {len(df):,})\n")

print("--- 5 Features (Knowable at decision moment) ---")
print("1. word_count (from dim_content): knowable because it is an intrinsic property of the page text.")
print("2. cpc (from dim_content): knowable because it is a static market property.")
print("3. gsc_avg_position (lagged): knowable because it is historical search performance prior to prediction.")
print("4. gsc_impressions (lagged): knowable because it summarizes traffic before the decision.")
print("5. ga4_sessions (lagged): knowable because it summarizes historical engagement.\n")

print("--- The Leakage Trap ---")
# To prove the trap, I show how a leaky current-day feature perfectly correlates with my target (clicks)
leaky_corr = df['gsc_impressions'].corr(df['gsc_clicks'])
print(f"Trap: I added current-day 'gsc_impressions' to predict current-day 'gsc_clicks'.")
print(f"Result: The correlation jumps to an insanely perfect {leaky_corr:.3f}!")
print("Why: Because current-day impressions happen at the exact same time as the clicks. It's predicting the present with the present.")
print("Action: I deliberately excluded all current-day metrics from my features and only use lagged (historical) metrics to keep the honest number.")


Loading March 2026 dataset from Hugging Face...
Loaded successfully!

Query 1: Grain verification (Page-Day)
Result: Max rows per client-content-date is 1 (if 1, grain holds perfectly!)

Query 2: Row count and Date Span
Result: 9,841,378 rows, from 2026-03-01 to 2026-03-31

Query 3: Availability (IS TRUE)
Result: 413,966 rows survive the GA4 availability filter (out of 9,841,378)

--- 5 Features (Knowable at decision moment) ---
1. word_count (from dim_content): knowable because it is an intrinsic property of the page text.
2. cpc (from dim_content): knowable because it is a static market property.
3. gsc_avg_position (lagged): knowable because it is historical search performance prior to prediction.
4. gsc_impressions (lagged): knowable because it summarizes traffic before the decision.
5. ga4_sessions (lagged): knowable because it summarizes historical engagement.

--- The Leakage Trap ---
Trap: I added current-day 'gsc_impressions' to predict current-day 'gsc_clicks'.
Result: The co

## 4. Data limits

* **Varying History Depth:** The history depth differs wildly per client. I cannot assume every client has full data back to 2025.
* **GA4 Zero-Fills:** Rows before a client's `ga4_data_start` have GA4 columns zero-filled with `ga4_data_available = FALSE`. These zeros do not mean "no engagement"; they mean "no tracking installed yet." I must filter on the flag.

In [26]:
# Acknowledged limits.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.